# LeetCode 44 — Wildcard Matching (Hard) bằng Python

Notebook này giảng từ căn bản bài toán:

```python
class Solution:
    def isMatch(self, s: str, p: str) -> bool:

```

**Mục tiêu sau khi học xong:**

1. Hiểu chính xác ý nghĩa của `'?'` và `'*'`, và thế nào là "khớp **toàn bộ**".
2. Biết viết **đệ quy** (recursion) từ con số 0: base case, bước đệ quy, cây lời gọi.
3. Hiểu vì sao đệ quy thuần bị chậm và sửa bằng **memoization** (ghi nhớ kết quả).
4. Chuyển sang **Dynamic Programming bảng** `dp[i][j]`, tự điền bảng bằng tay, rồi nén còn **1 chiều**.
5. Nắm cách **tham lam + quay lui** (greedy + backtracking) chỉ dùng bộ nhớ `O(1)`, và hiểu **vì sao chỉ cần nhớ dấu `'*'` gần nhất**.
6. Tự viết lại lời giải, test đối chiếu nhiều cách, và làm các biến thể.

Lộ trình: `Đệ quy thuần` → `Đệ quy + memo` → `DP bảng` → `DP 1 chiều` → `Greedy + backtracking`.

## 1. Đề bài (theo LeetCode)

Cho một chuỗi đầu vào `s` và một mẫu (pattern) `p`, hãy cài đặt phép so khớp mẫu đại diện (wildcard pattern matching) hỗ trợ `'?'` và `'*'`, trong đó:

- `'?'` khớp với **đúng một** ký tự bất kỳ.
- `'*'` khớp với **một dãy ký tự bất kỳ** (kể cả dãy rỗng).

Phép so khớp phải phủ **toàn bộ** chuỗi đầu vào (không phải khớp một phần).

**Example 1:**

```text
Input: s = "aa", p = "a"
Output: false
Explanation: "a" does not match the entire string "aa".
```

**Example 2:**

```text
Input: s = "aa", p = "*"
Output: true
Explanation: '*' matches any sequence.
```

**Example 3:**

```text
Input: s = "cb", p = "?a"
Output: false
Explanation: '?' matches 'c', but the second letter is 'a', which does not match 'b'.
```

**Constraints:**

- `0 <= s.length, p.length <= 2000`
- `s` chỉ chứa chữ cái tiếng Anh viết thường.
- `p` chỉ chứa chữ cái tiếng Anh viết thường, `'?'` hoặc `'*'`.

Lưu ý: `s` và `p` **có thể rỗng** — đây là nguồn của nhiều lỗi.

## 2. Hiểu đề bằng tay

Hãy coi `p` là một "khuôn". Mỗi ký tự trong khuôn phải "ăn" một phần của `s`, và tất cả các phần nối lại phải **đúng bằng** `s`:

| Ký tự trong `p` | Ăn bao nhiêu ký tự của `s`? |
|---|---|
| chữ cái, ví dụ `'a'` | đúng 1 ký tự, và ký tự đó phải là `'a'` |
| `'?'` | đúng 1 ký tự, ký tự gì cũng được |
| `'*'` | 0, 1, 2, ... ký tự, ký tự gì cũng được |

Vài ví dụ thêm (ngoài đề):

| `s` | `p` | Kết quả | Vì sao? |
|---|---|:---:|---|
| `"adceb"` | `"*a*b"` | `True` | `*`→`""`, `a`→`a`, `*`→`"dce"`, `b`→`b` |
| `"acdcb"` | `"a*c?b"` | `False` | đuôi `s` là `"dcb"`, nhưng `c?b` cần ký tự `c` đứng trước 2 ký tự cuối |
| `""` | `"***"` | `True` | mỗi `*` ăn chuỗi rỗng |
| `""` | `"?"` | `False` | `?` bắt buộc ăn đúng 1 ký tự |
| `"abc"` | `"a*"` | `True` | `*` ăn `"bc"` |

Khó khăn chính: **`'*'` ăn bao nhiêu thì đúng?** Ta không biết trước, nên phải "thử". Cell dưới liệt kê mọi cách `*` trong `"a*c"` có thể ăn khi `s = "abbc"`.

In [1]:
s = "abbc"
# p = "a*c": 'a' ăn s[0], 'c' ăn s[-1], còn '*' ăn phần ở giữa.
# Nhưng nếu chưa biết điều đó, ta thử cho '*' ăn từ vị trí 1, mỗi lần dài thêm 1 ký tự:
for k in range(0, len(s)):
    eaten = s[1:1 + k]          # đoạn '*' ăn
    rest = s[1 + k:]            # phần còn lại phải khớp với "c"
    print(f"'*' ăn {eaten!r:7} -> phần còn lại {rest!r:6} khớp 'c'? {rest == 'c'}")

'*' ăn ''      -> phần còn lại 'bbc'  khớp 'c'? False
'*' ăn 'b'     -> phần còn lại 'bc'   khớp 'c'? False
'*' ăn 'bb'    -> phần còn lại 'c'    khớp 'c'? True
'*' ăn 'bbc'   -> phần còn lại ''     khớp 'c'? False


Chỉ đúng **một** lựa chọn (`'*'` ăn `"bb"`) dẫn tới thành công. Với nhiều dấu `'*'`, số tổ hợp lựa chọn tăng rất nhanh — đó là lý do bài này xếp Hard.

**Đừng nhầm với bài 10 — Regular Expression Matching:**

| | Bài 44 (Wildcard) | Bài 10 (Regex) |
|---|---|---|
| Một ký tự bất kỳ | `'?'` | `'.'` |
| Dấu `'*'` | tự nó khớp **dãy bất kỳ** | lặp lại **ký tự đứng trước** nó 0+ lần |
| Ví dụ `"*"` khớp `"xyz"` | `True` | không hợp lệ |

Và "khớp toàn bộ" nghĩa là `p = "a"` **không** khớp `s = "aa"` (Example 1), dù `"a"` có xuất hiện trong `"aa"`.

## 3. Kiến thức nền

### 3.1. So khớp một ký tự

Với ký tự **không phải** `'*'`, việc kiểm tra rất đơn giản: ký tự mẫu `pc` khớp ký tự chuỗi `sc` khi `pc == '?'` hoặc `pc == sc`.

In [2]:
def char_ok(sc: str, pc: str) -> bool:
    """pc là '?' hoặc một chữ cái (KHÔNG phải '*')."""
    return pc == '?' or pc == sc

print(char_ok('a', 'a'))   # True : giống nhau
print(char_ok('b', '?'))   # True : '?' khớp mọi ký tự
print(char_ok('b', 'a'))   # False: khác nhau

True
True
False


### 3.2. Đệ quy (recursion) là gì?

**Đệ quy** = một hàm **tự gọi lại chính nó** trên một bài toán **nhỏ hơn**. Mỗi hàm đệ quy cần 2 phần:

1. **Base case** (trường hợp cơ sở): bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Recursive case** (bước đệ quy): làm một chút việc, rồi giao phần còn lại cho lời gọi nhỏ hơn.

Ví dụ mini, rất gần với bài của ta: **kiểm tra hai chuỗi `s` và `p` có bằng nhau không**, xét từ vị trí `i`:

- Base case: nếu `i` đã đi hết cả hai chuỗi → bằng nhau.
- Nếu một chuỗi hết mà chuỗi kia chưa → khác nhau.
- Bước đệ quy: nếu `s[i] == p[i]` thì câu trả lời là "phần còn lại từ `i + 1` có bằng nhau không?".

Mỗi lần gọi hàm, Python đặt một "khung" lên **call stack** (ngăn xếp lời gọi). Tham số `depth` dưới đây chỉ dùng để thụt lề khi in, cho ta nhìn thấy các khung chồng lên nhau.

In [3]:
def equal_rec(s: str, p: str, i: int = 0, depth: int = 0) -> bool:
    pad = "  " * depth
    print(f"{pad}equal_rec(i={i})")
    if i == len(s) and i == len(p):      # base case 1: cả hai cùng hết
        return True
    if i == len(s) or i == len(p):       # base case 2: một bên hết trước
        return False
    if s[i] != p[i]:                     # khác ngay tại i
        return False
    return equal_rec(s, p, i + 1, depth + 1)   # bước đệ quy: bài nhỏ hơn

print("Kết quả:", equal_rec("abc", "abc"))
print("Kết quả:", equal_rec("abc", "abd"))

equal_rec(i=0)
  equal_rec(i=1)
    equal_rec(i=2)
      equal_rec(i=3)
Kết quả: True
equal_rec(i=0)
  equal_rec(i=1)
    equal_rec(i=2)
Kết quả: False


### 3.3. Dùng chỉ số `i`, `j` thay vì cắt chuỗi

Khi giải đệ quy, ta luôn hỏi: "**phần còn lại** của `s` có khớp **phần còn lại** của `p` không?". Phần còn lại tính từ vị trí `i` là `s[i:]` — gọi là **hậu tố** (suffix).

Ta có thể truyền chuỗi đã cắt `s[1:]` vào lời gọi đệ quy, nhưng mỗi lần cắt Python **tạo chuỗi mới** (tốn `O(độ dài)` thời gian và bộ nhớ). Cách gọn hơn: chỉ truyền **chỉ số** `i` (vị trí trong `s`) và `j` (vị trí trong `p`).

In [4]:
s = "abc"
for i in range(len(s) + 1):
    print(f"i = {i}: s[i:] = {s[i:]!r}")
# Khi i == len(s), hậu tố là chuỗi rỗng "".

i = 0: s[i:] = 'abc'
i = 1: s[i:] = 'bc'
i = 2: s[i:] = 'c'
i = 3: s[i:] = ''


## 4. Cách 1 — Đệ quy thuần (brute force)

### 4.1. Định nghĩa bài toán con

Gọi `match(i, j)` = "hậu tố `s[i:]` có khớp **toàn bộ** với hậu tố `p[j:]` không?". Đáp án của đề là `match(0, 0)`.

Xét ký tự mẫu đầu tiên còn lại, `p[j]`:

| Trường hợp | Trả về |
|---|---|
| **Base case:** `j == len(p)` (mẫu đã hết) | `i == len(s)` — chuỗi cũng phải hết |
| `p[j] == '*'` | `match(i, j + 1)` **hoặc** `match(i + 1, j)` (cần `i < len(s)`) |
| `p[j]` là `'?'` hoặc chữ cái | `i < len(s)` **và** `char_ok(s[i], p[j])` **và** `match(i + 1, j + 1)` |

### 4.2. Vì sao `'*'` chỉ cần **2 lựa chọn**?

Ta không cần thử `'*'` ăn 0, 1, 2, ..., k ký tự một cách riêng lẻ. Chỉ cần 2 lựa chọn:

- **Lựa chọn A — `'*'` ăn rỗng:** bỏ `'*'` đi, xét `match(i, j + 1)`.
- **Lựa chọn B — `'*'` ăn thêm 1 ký tự `s[i]`, nhưng vẫn ở lại:** xét `match(i + 1, j)`. Vì `j` không đổi, lần sau `'*'` lại được chọn A hoặc B.

"Ăn k ký tự" = chọn B đúng k lần rồi chọn A. Nên 2 lựa chọn này phủ hết mọi khả năng.

In [5]:
def wildcard_recursive(s: str, p: str) -> bool:
    def match(i: int, j: int) -> bool:
        if j == len(p):                          # mẫu hết
            return i == len(s)
        if p[j] == '*':
            # A: '*' ăn rỗng            B: '*' ăn s[i] và ở lại
            return match(i, j + 1) or (i < len(s) and match(i + 1, j))
        # p[j] là '?' hoặc chữ cái: phải ăn đúng 1 ký tự
        return i < len(s) and char_ok(s[i], p[j]) and match(i + 1, j + 1)

    return match(0, 0)

examples = [("aa", "a"), ("aa", "*"), ("cb", "?a"), ("adceb", "*a*b"), ("acdcb", "a*c?b"), ("", "***"), ("", "?")]
for s, p in examples:
    print(f"s={s!r:8} p={p!r:8} -> {wildcard_recursive(s, p)}")

s='aa'     p='a'      -> False
s='aa'     p='*'      -> True
s='cb'     p='?a'     -> False
s='adceb'  p='*a*b'   -> True
s='acdcb'  p='a*c?b'  -> False
s=''       p='***'    -> True
s=''       p='?'      -> False


### 4.3. Trace cây lời gọi: `s = "aab"`, `p = "*ab"`

Ta chọn ví dụ này vì lựa chọn A của `'*'` (ăn rỗng) **thất bại**, buộc phải **quay lui** (backtrack) sang lựa chọn B.

| Lời gọi | Đang xét | Quyết định |
|---|---|---|
| `match(0,0)` | `p[0]='*'` | thử A trước: `match(0,1)` |
| `match(0,1)` | `s[0]='a'`, `p[1]='a'` | khớp → `match(1,2)` |
| `match(1,2)` | `s[1]='a'`, `p[2]='b'` | không khớp → `False` |
| (quay về `match(0,0)`) | A thất bại | thử B: `match(1,0)` |
| `match(1,0)` | `p[0]='*'` | thử A: `match(1,1)` |
| `match(1,1)` | `'a'` vs `'a'` | khớp → `match(2,2)` |
| `match(2,2)` | `'b'` vs `'b'` | khớp → `match(3,3)` |
| `match(3,3)` | mẫu hết, chuỗi hết | `True` |

Hàm `trace_recursive` in đúng cây này: mỗi mức thụt lề là một tầng trên call stack.

In [6]:
def trace_recursive(s: str, p: str) -> bool:
    def match(i: int, j: int, depth: int) -> bool:
        pad = "  " * depth
        rest_s, rest_p = s[i:], p[j:]
        print(f"{pad}match({i},{j})  s[i:]={rest_s!r:6} p[j:]={rest_p!r}")
        if j == len(p):
            ans = i == len(s)
        elif p[j] == '*':
            ans = match(i, j + 1, depth + 1) or (i < len(s) and match(i + 1, j, depth + 1))
        else:
            ans = i < len(s) and char_ok(s[i], p[j]) and match(i + 1, j + 1, depth + 1)
        print(f"{pad}-> {ans}")
        return ans

    return match(0, 0, 0)

print("Kết quả cuối:", trace_recursive("aab", "*ab"))

match(0,0)  s[i:]='aab'  p[j:]='*ab'
  match(0,1)  s[i:]='aab'  p[j:]='ab'
    match(1,2)  s[i:]='ab'   p[j:]='b'
    -> False
  -> False
  match(1,0)  s[i:]='ab'   p[j:]='*ab'
    match(1,1)  s[i:]='ab'   p[j:]='ab'
      match(2,2)  s[i:]='b'    p[j:]='b'
        match(3,3)  s[i:]=''     p[j:]=''
        -> True
      -> True
    -> True
  -> True
-> True
Kết quả cuối: True


### 4.4. Cách 1 vướng ở đâu?

Mỗi `'*'` tạo ra **2 nhánh**. Khi đáp án là `False`, hàm phải đi hết mọi nhánh mới dám kết luận. Thử với `s = "aaa...a"` (k chữ `a`) và `p = "*a*a*a*a*b"` (đáp án luôn `False` vì `s` không có `b`).

Để đếm số lời gọi, ta dùng biến `calls` ở hàm ngoài và khai báo `nonlocal calls` trong hàm trong — từ khóa `nonlocal` cho phép hàm con **sửa** biến của hàm bao nó.

In [7]:
def count_calls_recursive(s: str, p: str):
    calls = 0
    def match(i, j):
        nonlocal calls
        calls += 1
        if j == len(p):
            return i == len(s)
        if p[j] == '*':
            return match(i, j + 1) or (i < len(s) and match(i + 1, j))
        return i < len(s) and char_ok(s[i], p[j]) and match(i + 1, j + 1)
    return match(0, 0), calls

p = "*a*a*a*a*b"
for k in [5, 10, 15, 20, 25]:
    s = "a" * k
    result, calls = count_calls_recursive(s, p)
    distinct_states = (len(s) + 1) * (len(p) + 1)
    print(f"k={k:2}: kết quả={result}, số lời gọi={calls:7}, số cặp (i,j) khác nhau chỉ có={distinct_states}")

k= 5: kết quả=False, số lời gọi=    124, số cặp (i,j) khác nhau chỉ có=66
k=10: kết quả=False, số lời gọi=   2046, số cặp (i,j) khác nhau chỉ có=121
k=15: kết quả=False, số lời gọi=  13768, số cặp (i,j) khác nhau chỉ có=176
k=20: kết quả=False, số lời gọi=  55790, số cặp (i,j) khác nhau chỉ có=231
k=25: kết quả=False, số lời gọi= 167362, số cặp (i,j) khác nhau chỉ có=286


### 4.5. Phát hiện quan trọng: các bài toán con bị **lặp lại**

Số lời gọi tăng vọt (theo lũy thừa bậc cao của `k`; mẫu càng nhiều `'*'` thì bậc càng cao — tức tăng theo hàm mũ của số dấu `'*'`), nhưng số **cặp `(i, j)` khác nhau** chỉ có `(len(s)+1) × (len(p)+1)`. Nghĩa là cùng một câu hỏi `match(i, j)` bị hỏi đi hỏi lại rất nhiều lần.

Ta dùng `Counter` (trong `collections`) — một dict đặc biệt dùng để **đếm số lần xuất hiện** — để xem cặp nào bị gọi nhiều nhất.

In [8]:
from collections import Counter

def count_states(s: str, p: str) -> Counter:
    seen = Counter()
    def match(i, j):
        seen[(i, j)] += 1
        if j == len(p):
            return i == len(s)
        if p[j] == '*':
            return match(i, j + 1) or (i < len(s) and match(i + 1, j))
        return i < len(s) and char_ok(s[i], p[j]) and match(i + 1, j + 1)
    match(0, 0)
    return seen

seen = count_states("a" * 10, "*a*a*a*a*b")
print("Số cặp (i,j) khác nhau được gọi:", len(seen))
print("Tổng số lời gọi:", sum(seen.values()))
print("5 cặp bị gọi lặp nhiều nhất:", seen.most_common(5))

Số cặp (i,j) khác nhau được gọi: 90
Tổng số lời gọi: 2046
5 cặp bị gọi lặp nhiều nhất: [((10, 8), 210), ((10, 9), 210), ((9, 8), 126), ((9, 9), 126), ((10, 6), 120)]


## 5. Cách 2 — Đệ quy + memoization

### 5.1. Memoization là gì? (ví dụ mini với Fibonacci)

**Memoization** (ghi nhớ) = lưu kết quả của mỗi bài toán con vào một dict `memo` ngay khi tính xong. Lần sau gặp lại, **tra dict** thay vì tính lại.

Ví dụ kinh điển: `fib(n) = fib(n-1) + fib(n-2)`. Bản thuần gọi lại `fib(3)`, `fib(2)`... rất nhiều lần.

In [9]:
calls_naive = 0
def fib_naive(n):
    global calls_naive
    calls_naive += 1
    return n if n < 2 else fib_naive(n - 1) + fib_naive(n - 2)

calls_memo = 0
memo = {}
def fib_memo(n):
    global calls_memo
    calls_memo += 1
    if n in memo:                 # đã tính rồi -> trả luôn
        return memo[n]
    ans = n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)
    memo[n] = ans                 # lưu lại trước khi trả
    return ans

print("fib(25) thuần :", fib_naive(25), "| số lời gọi:", calls_naive)
print("fib(25) memo  :", fib_memo(25), "| số lời gọi:", calls_memo)

fib(25) thuần : 75025 | số lời gọi: 242785
fib(25) memo  : 75025 | số lời gọi: 49


### 5.2. Áp dụng vào Wildcard Matching

Giữ nguyên 3 luật của Cách 1, chỉ thêm 2 dòng: **tra `memo` đầu hàm** và **lưu `memo` trước khi trả**. Khóa (key) của dict là bộ `(i, j)` — trong Python, tuple dùng làm key được vì nó không thay đổi được (immutable).

Python còn có sẵn `@functools.lru_cache(maxsize=None)` (hoặc `@functools.cache`) tự làm việc này, nhưng tự viết `memo` giúp ta thấy rõ cơ chế.

In [10]:
def wildcard_memo(s: str, p: str) -> bool:
    memo = {}

    def match(i: int, j: int) -> bool:
        if (i, j) in memo:
            return memo[(i, j)]
        if j == len(p):
            ans = i == len(s)
        elif p[j] == '*':
            ans = match(i, j + 1) or (i < len(s) and match(i + 1, j))
        else:
            ans = i < len(s) and char_ok(s[i], p[j]) and match(i + 1, j + 1)
        memo[(i, j)] = ans
        return ans

    return match(0, 0)

for s, p in examples:
    print(f"s={s!r:8} p={p!r:8} -> {wildcard_memo(s, p)}")

# Ca trước đây tốn hàng trăm nghìn lời gọi giờ chạy tức thì:
print(wildcard_memo("a" * 25, "*a*a*a*a*b"))

s='aa'     p='a'      -> False
s='aa'     p='*'      -> True
s='cb'     p='?a'     -> False
s='adceb'  p='*a*b'   -> True
s='acdcb'  p='a*c?b'  -> False
s=''       p='***'    -> True
s=''       p='?'      -> False
False


### 5.3. Độ phức tạp và một cái bẫy

Gọi `m = len(s)`, `n = len(p)`.

- **Time:** `O(m · n)` — có tối đa `(m+1)(n+1)` cặp `(i, j)`, mỗi cặp tính một lần với `O(1)` việc.
- **Space:** `O(m · n)` cho `memo`, cộng `O(m + n)` cho độ sâu call stack.

**Cái bẫy:** Python giới hạn độ sâu đệ quy mặc định khoảng `1000` tầng. Với constraints `m, n ≤ 2000`, chuỗi lời gọi `match(0,0) → match(1,1) → ...` có thể sâu tới `m + n ≈ 4000` tầng → `RecursionError`. Ta có thể nâng giới hạn bằng `sys.setrecursionlimit`, nhưng cách bền vững hơn là **bỏ đệ quy**: tính bảng từ dưới lên (Cách 3).

In [11]:
import sys
print("Giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

long_s = "a" * 2000
long_p = "a" * 2000
try:
    print(wildcard_memo(long_s, long_p))
except RecursionError as e:
    print("RecursionError:", e)

Giới hạn đệ quy hiện tại: 1000
RecursionError: maximum recursion depth exceeded


## 6. Cách 3 — Dynamic Programming bảng (bottom-up)

### 6.1. Từ "hỏi từ trên xuống" sang "điền bảng từ dưới lên"

Memo là **top-down**: hỏi `match(0,0)` rồi đệ quy xuống bài nhỏ. **Bottom-up** làm ngược lại: tính sẵn **mọi** bài nhỏ theo thứ tự, lưu vào bảng, không cần đệ quy.

Với bảng, cách tự nhiên nhất là nhìn theo **tiền tố** (prefix — phần đầu của chuỗi):

> `dp[i][j]` = "**`i` ký tự đầu** của `s` (tức `s[:i]`) có khớp **`j` ký tự đầu** của `p` (tức `p[:j]`) không?"

Bảng có `(m+1)` hàng × `(n+1)` cột (thêm hàng/cột `0` cho chuỗi rỗng). Đáp án là `dp[m][n]`. Ký tự **cuối** của tiền tố là `s[i-1]` và `p[j-1]` (chú ý `-1`).

Ba luật giống hệt Cách 1, chỉ là nhìn từ **cuối** tiền tố:

| Trường hợp | Công thức | Ý nghĩa |
|---|---|---|
| `dp[0][0]` | `True` | rỗng khớp rỗng |
| `dp[i][0]`, `i ≥ 1` | `False` | mẫu rỗng không khớp chuỗi khác rỗng |
| `dp[0][j]` | `dp[0][j-1]` nếu `p[j-1] == '*'`, ngược lại `False` | chuỗi rỗng chỉ khớp toàn `'*'` |
| `p[j-1] == '*'` | `dp[i][j-1]` **or** `dp[i-1][j]` | `'*'` ăn rỗng **or** `'*'` ăn thêm `s[i-1]` |
| `p[j-1]` là `'?'`/chữ khớp `s[i-1]` | `dp[i-1][j-1]` | bỏ cặp ký tự cuối |
| còn lại | `False` | ký tự cuối không khớp |

Mỗi ô chỉ cần ô **bên trái**, ô **bên trên**, ô **chéo trên-trái** → điền từng hàng, từ trái sang phải là đủ.

### 6.2. Tạo bảng 2 chiều trong Python (và một lỗi kinh điển)

Cách đúng: **list comprehension** — tạo **mỗi hàng là một list mới**:

```python
dp = [[False] * (n + 1) for _ in range(m + 1)]
```

Cách **sai** thường gặp: `[[False] * (n + 1)] * (m + 1)` — nhân list ngoài chỉ **sao chép tham chiếu**, tất cả các hàng là **cùng một list**. Sửa một ô là sửa cả cột!

In [12]:
m, n = 2, 3
good = [[False] * (n + 1) for _ in range(m + 1)]
bad = [[False] * (n + 1)] * (m + 1)

good[0][0] = True
bad[0][0] = True
print("good:", good)
print("bad :", bad)          # cả 3 hàng đều đổi!
print("Các hàng của bad là cùng một list?", bad[0] is bad[1])

good: [[True, False, False, False], [False, False, False, False], [False, False, False, False]]
bad : [[True, False, False, False], [True, False, False, False], [True, False, False, False]]
Các hàng của bad là cùng một list? True


In [13]:
def wildcard_dp_table(s: str, p: str) -> list:
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for j in range(1, n + 1):                 # hàng 0: chuỗi rỗng
        if p[j - 1] == '*':
            dp[0][j] = dp[0][j - 1]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == '*':
                dp[i][j] = dp[i][j - 1] or dp[i - 1][j]
            elif char_ok(s[i - 1], p[j - 1]):
                dp[i][j] = dp[i - 1][j - 1]
            # còn lại: giữ False
    return dp


def wildcard_dp(s: str, p: str) -> bool:
    return wildcard_dp_table(s, p)[len(s)][len(p)]


for s, p in examples:
    print(f"s={s!r:8} p={p!r:8} -> {wildcard_dp(s, p)}")
print("Ca dài 2000 ký tự:", wildcard_dp("a" * 2000, "a" * 1999 + "?"))

s='aa'     p='a'      -> False
s='aa'     p='*'      -> True
s='cb'     p='?a'     -> False
s='adceb'  p='*a*b'   -> True
s='acdcb'  p='a*c?b'  -> False
s=''       p='***'    -> True
s=''       p='?'      -> False


Ca dài 2000 ký tự: True


### 6.3. Trace bảng: `s = "adceb"`, `p = "*a*b"`

Hàng = tiền tố của `s`, cột = tiền tố của `p` (đầu cột ghi ký tự cuối của tiền tố). `T` = True, `.` = False.

| `i` \ `j` | 0: `ε` | 1: `*` | 2: `a` | 3: `*` | 4: `b` |
|---|:-:|:-:|:-:|:-:|:-:|
| **0**: `ε` | T | T | . | . | . |
| **1**: `a` | . | T | T | T | . |
| **2**: `d` | . | T | . | T | . |
| **3**: `c` | . | T | . | T | . |
| **4**: `e` | . | T | . | T | . |
| **5**: `b` | . | T | . | T | T |

Đọc vài ô tiêu biểu:

- `dp[0][1] = T`: `""` khớp `"*"` vì `'*'` ăn rỗng (luật hàng 0).
- `dp[1][2] = T`: `p[1] = 'a'` khớp `s[0] = 'a'` → lấy ô chéo `dp[0][1] = T`.
- `dp[2][3] = T`: `p[2] = '*'` → `dp[2][2] (.)` **or** `dp[1][3] (T)` = T. Đây là `'*'` thứ hai bắt đầu "ăn" chữ `d`.
- Cột `'*'` thứ hai (cột 3) toàn `T` từ hàng 1 trở xuống: một khi đã khớp, `'*'` **lan xuống** mãi (luật `dp[i-1][j]`).
- `dp[5][4] = T`: `p[3] = 'b'` khớp `s[4] = 'b'` → lấy chéo `dp[4][3] = T`. Đáp án `True`.

In [14]:
def print_dp_table(s: str, p: str) -> None:
    dp = wildcard_dp_table(s, p)
    header = "      " + "  ".join(["ε"] + list(p))     # ε = chuỗi rỗng
    print(header)
    for i in range(len(s) + 1):
        label = "ε" if i == 0 else s[i - 1]
        row = "  ".join("T" if v else "." for v in dp[i])
        print(f"{i} {label}   {row}")
    print("Đáp án dp[m][n] =", dp[len(s)][len(p)])

print_dp_table("adceb", "*a*b")
print()
print_dp_table("acdcb", "a*c?b")

      ε  *  a  *  b
0 ε   T  T  .  .  .
1 a   .  T  T  T  .
2 d   .  T  .  T  .
3 c   .  T  .  T  .
4 e   .  T  .  T  .
5 b   .  T  .  T  T
Đáp án dp[m][n] = True

      ε  a  *  c  ?  b
0 ε   T  .  .  .  .  .
1 a   .  T  T  .  .  .
2 c   .  .  T  T  .  .
3 d   .  .  T  .  T  .
4 c   .  .  T  T  .  .
5 b   .  .  T  .  T  .
Đáp án dp[m][n] = False


Ở bảng thứ hai (`"acdcb"`, `"a*c?b"`), ô cuối là `.` (False): để `'b'` cuối khớp, cần `dp[4][4]` (tiền tố `"acdc"` khớp `"a*c?"`) là `T`, nhưng `"a*c?"` đòi ký tự **áp chót** của `"acdc"` là `c` — thực tế là `d`.

**Độ phức tạp Cách 3** (`m = len(s)`, `n = len(p)`):

- **Time:** `O(m · n)` — điền `(m+1)(n+1)` ô, mỗi ô `O(1)`.
- **Space:** `O(m · n)` cho cả bảng. Với `m = n = 2000` là khoảng 4 triệu ô.
- Không đệ quy → không lo `RecursionError`.

## 7. Cách 4 — DP 1 chiều (tiết kiệm bộ nhớ)

Nhìn lại công thức: ô hàng `i` chỉ dùng **hàng `i-1`** (ô trên, ô chéo) và **chính hàng `i`** (ô trái). Các hàng cũ hơn không bao giờ dùng lại.

Vậy chỉ cần giữ 2 hàng: `prev` (hàng trên) và `cur` (hàng đang điền). Xong một hàng thì `prev = cur`.

Chú ý: `cur[0]` luôn là `False` với `i ≥ 1` (mẫu rỗng không khớp chuỗi khác rỗng) — tạo `cur` mới toàn `False` là đã đúng.

In [15]:
def wildcard_dp_1d(s: str, p: str, show: bool = False) -> bool:
    n = len(p)
    prev = [False] * (n + 1)
    prev[0] = True
    for j in range(1, n + 1):
        prev[j] = prev[j - 1] and p[j - 1] == '*'
    if show:
        print("hàng 0  :", "".join("T" if v else "." for v in prev))
    for i in range(1, len(s) + 1):
        cur = [False] * (n + 1)
        for j in range(1, n + 1):
            if p[j - 1] == '*':
                cur[j] = cur[j - 1] or prev[j]       # trái (cùng hàng) or trên
            elif char_ok(s[i - 1], p[j - 1]):
                cur[j] = prev[j - 1]                 # chéo trên-trái
        prev = cur
        if show:
            print(f"hàng {i} {s[i-1]!r}:", "".join("T" if v else "." for v in prev))
    return prev[n]

print(wildcard_dp_1d("adceb", "*a*b", show=True))
for s, p in examples:
    assert wildcard_dp_1d(s, p) == wildcard_dp(s, p)
print("Khớp với Cách 3 trên mọi ví dụ.")

hàng 0  : TT...
hàng 1 'a': .TTT.
hàng 2 'd': .T.T.
hàng 3 'c': .T.T.
hàng 4 'e': .T.T.
hàng 5 'b': .T.TT
True
Khớp với Cách 3 trên mọi ví dụ.


Mỗi dòng in ra chính là một hàng của bảng ở mục 6.3 — ta chỉ không giữ lại các hàng cũ.

**Độ phức tạp Cách 4:** Time `O(m · n)`, Space `O(n)`.

(Có thể nén thêm còn **một** mảng duy nhất, nhưng khi đó ô chéo `prev[j-1]` bị ghi đè trước khi dùng nên phải lưu tạm vào một biến — dễ sai, không cần thiết cho người mới.)

Câu hỏi tự nhiên: có thể xuống `O(1)` bộ nhớ không? Có — nhưng phải đổi hẳn cách nghĩ.

## 8. Cách 5 — Tham lam + quay lui với dấu `'*'` gần nhất

### 8.1. Nhìn mẫu như các "đoạn chữ" ngăn bởi `'*'`

Mẫu `"*a*b"` thực chất là: `[đoạn rỗng] * [a] * [b]`. Giữa hai dấu `'*'` là một **đoạn cố định** (chỉ gồm chữ và `'?'`) — đoạn này có độ dài cố định, phải khớp liền một khúc trong `s`.

Còn `'*'` chỉ là "keo" co giãn lấp khoảng trống giữa các đoạn.

In [16]:
for p in ["*a*b", "a*c?b", "ab*cd?i*de", "***"]:
    print(f"{p!r:14} -> các đoạn: {p.split('*')}")
# Đoạn đầu phải khớp ngay đầu s, đoạn cuối phải khớp ở cuối s,
# các đoạn giữa chỉ cần xuất hiện theo đúng thứ tự.

'*a*b'         -> các đoạn: ['', 'a', 'b']
'a*c?b'        -> các đoạn: ['a', 'c?b']
'ab*cd?i*de'   -> các đoạn: ['ab', 'cd?i', 'de']
'***'          -> các đoạn: ['', '', '', '']


### 8.2. Ý tưởng tham lam (greedy)

**Tham lam** = mỗi bước chọn phương án "trông có vẻ tốt nhất ngay lúc đó" và không xét lại toàn bộ. Ở đây:

> Cho mỗi `'*'` ăn **ít nhất có thể** (bắt đầu từ rỗng), để đoạn chữ phía sau được đặt **sớm nhất có thể**. Nếu bị kẹt, chỉ quay lại **dấu `'*'` gần nhất** và cho nó ăn thêm **1** ký tự.

Ta dùng **hai con trỏ** (two pointers) — đơn giản là hai biến chỉ số chạy trên hai chuỗi: `i` trên `s`, `j` trên `p`. Thêm 2 biến nhớ:

- `star`: vị trí của dấu `'*'` gần nhất đã gặp trong `p` (`-1` nếu chưa có).
- `match`: vị trí trong `s` ngay sau phần mà `'*'` đó đang ăn — tức `'*'` đang ăn `s[vị trí bắt đầu : match]`.

Vòng lặp `while i < len(s)` có 4 nhánh, **xét theo thứ tự**:

| Nhánh | Điều kiện | Hành động |
|---|---|---|
| 1. Khớp | `j < n` và `p[j]` là `'?'` hoặc bằng `s[i]` | `i += 1`, `j += 1` |
| 2. Gặp `'*'` | `j < n` và `p[j] == '*'` | `star = j`, `match = i`, `j += 1` (thử `'*'` ăn rỗng) |
| 3. Kẹt, có `'*'` để lùi | `star != -1` | `j = star + 1`, `match += 1`, `i = match` (cho `'*'` ăn thêm 1) |
| 4. Kẹt, không có `'*'` | còn lại | `return False` |

Sau vòng lặp, `s` đã hết; phần còn lại của `p` chỉ được phép toàn `'*'` (chúng ăn rỗng).

Nhánh 1 không bao giờ "nuốt nhầm" `'*'` vì `s` chỉ chứa chữ thường, nên `p[j] == s[i]` không thể đúng khi `p[j] == '*'`.

In [17]:
def wildcard_greedy(s: str, p: str) -> bool:
    m, n = len(s), len(p)
    i = j = 0
    star = -1          # vị trí '*' gần nhất trong p
    match = 0          # '*' đó đang ăn s[...:match]
    while i < m:
        if j < n and (p[j] == '?' or p[j] == s[i]):    # nhánh 1
            i += 1
            j += 1
        elif j < n and p[j] == '*':                     # nhánh 2
            star = j
            match = i
            j += 1
        elif star != -1:                                # nhánh 3
            j = star + 1
            match += 1
            i = match
        else:                                           # nhánh 4
            return False
    while j < n and p[j] == '*':                        # '*' thừa ở cuối ăn rỗng
        j += 1
    return j == n

for s, p in examples:
    print(f"s={s!r:8} p={p!r:8} -> {wildcard_greedy(s, p)}")

s='aa'     p='a'      -> False
s='aa'     p='*'      -> True
s='cb'     p='?a'     -> False
s='adceb'  p='*a*b'   -> True
s='acdcb'  p='a*c?b'  -> False
s=''       p='***'    -> True
s=''       p='?'      -> False


### 8.3. Trace: `s = "abcbcd"`, `p = "a*bcd"`

Ví dụ này buộc phải quay lui **hai lần**: `'*'` thử ăn rỗng, rồi `"b"`, rồi `"bc"` mới thành công. Cột `i`, `j`, `s[i]`, `p[j]` là trạng thái **trước** bước; cột `star`, `match` là giá trị **sau** bước.

| Bước | `i` | `j` | `s[i]` | `p[j]` | `star` (sau bước) | `match` (sau bước) | Hành động |
|---|---|---|---|---|---|---|---|
| B1 | 0 | 0 | `a` | `a` | -1 | 0 | khớp -> i=1, j=1 |
| B2 | 1 | 1 | `b` | `*` | 1 | 1 | gặp '*': nhớ star=1, match=1; thử ăn rỗng, j=2 |
| B3 | 1 | 2 | `b` | `b` | 1 | 1 | khớp -> i=2, j=3 |
| B4 | 2 | 3 | `c` | `c` | 1 | 1 | khớp -> i=3, j=4 |
| B5 | 3 | 4 | `b` | `d` | 1 | 2 | kẹt -> lùi: '*' ăn 'b', i=2, j=2 |
| B6 | 2 | 2 | `c` | `b` | 1 | 3 | kẹt -> lùi: '*' ăn 'bc', i=3, j=2 |
| B7 | 3 | 2 | `b` | `b` | 1 | 3 | khớp -> i=4, j=3 |
| B8 | 4 | 3 | `c` | `c` | 1 | 3 | khớp -> i=5, j=4 |
| B9 | 5 | 4 | `d` | `d` | 1 | 3 | khớp -> i=6, j=5 |

Kết thúc vòng lặp: `i = 6 = len(s)`, `j = 5 = len(p)` → **True** (`'*'` đã ăn `"bc"`).

Thêm một ca `False`: `s = "acdcb"`, `p = "a*c?b"`.

| Bước | `i` | `j` | `s[i]` | `p[j]` | `star` (sau bước) | `match` (sau bước) | Hành động |
|---|---|---|---|---|---|---|---|
| B1 | 0 | 0 | `a` | `a` | -1 | 0 | khớp -> i=1, j=1 |
| B2 | 1 | 1 | `c` | `*` | 1 | 1 | gặp '*': nhớ star=1, match=1; thử ăn rỗng, j=2 |
| B3 | 1 | 2 | `c` | `c` | 1 | 1 | khớp -> i=2, j=3 |
| B4 | 2 | 3 | `d` | `?` | 1 | 1 | khớp -> i=3, j=4 |
| B5 | 3 | 4 | `c` | `b` | 1 | 2 | kẹt -> lùi: '*' ăn 'c', i=2, j=2 |
| B6 | 2 | 2 | `d` | `c` | 1 | 3 | kẹt -> lùi: '*' ăn 'cd', i=3, j=2 |
| B7 | 3 | 2 | `c` | `c` | 1 | 3 | khớp -> i=4, j=3 |
| B8 | 4 | 3 | `b` | `?` | 1 | 3 | khớp -> i=5, j=4 |

Kết thúc vòng lặp: `s` đã hết (`i = 5`) nhưng `j = 4`, `p[4] = 'b'` không phải `'*'` → **False**.

In [18]:
def greedy_steps(s: str, p: str):
    """Chạy giống wildcard_greedy nhưng ghi lại từng bước; trả về (kết quả, danh sách bước).
    Biến star_begin chỉ dùng để in: vị trí trong s nơi '*' gần nhất bắt đầu ăn."""
    m, n = len(s), len(p)
    i = j = 0
    star, match = -1, 0
    star_begin = 0
    steps = []
    while i < m:
        sc = s[i]
        pc = p[j] if j < n else "hết"
        before = (i, j, sc, pc)
        if j < n and (p[j] == '?' or p[j] == s[i]):
            i, j = i + 1, j + 1
            action = f"khớp -> i={i}, j={j}"
        elif j < n and p[j] == '*':
            star, match, star_begin = j, i, i
            j += 1
            action = f"gặp '*': nhớ star={star}, match={match}; thử ăn rỗng, j={j}"
        elif star != -1:
            j = star + 1
            match += 1
            i = match
            action = f"kẹt -> lùi: '*' ăn {s[star_begin:match]!r}, i={i}, j={j}"
        else:
            steps.append((*before, star, match, "kẹt, không có '*' -> False"))
            return False, steps
        steps.append((*before, star, match, action))
    while j < n and p[j] == '*':
        j += 1
    return j == n, steps


def trace_greedy(s: str, p: str) -> bool:
    print(f"s = {s!r}, p = {p!r}")
    result, steps = greedy_steps(s, p)
    for k, (i, j, sc, pc, star, match, action) in enumerate(steps, 1):
        print(f"B{k}: i={i} j={j} s[i]={sc!r} p[j]={pc!r:5} | {action}")
    print("Kết quả:", result)
    return result

_ = trace_greedy("abcbcd", "a*bcd")
print()
_ = trace_greedy("acdcb", "a*c?b")

s = 'abcbcd', p = 'a*bcd'
B1: i=0 j=0 s[i]='a' p[j]='a'   | khớp -> i=1, j=1
B2: i=1 j=1 s[i]='b' p[j]='*'   | gặp '*': nhớ star=1, match=1; thử ăn rỗng, j=2
B3: i=1 j=2 s[i]='b' p[j]='b'   | khớp -> i=2, j=3
B4: i=2 j=3 s[i]='c' p[j]='c'   | khớp -> i=3, j=4
B5: i=3 j=4 s[i]='b' p[j]='d'   | kẹt -> lùi: '*' ăn 'b', i=2, j=2
B6: i=2 j=2 s[i]='c' p[j]='b'   | kẹt -> lùi: '*' ăn 'bc', i=3, j=2
B7: i=3 j=2 s[i]='b' p[j]='b'   | khớp -> i=4, j=3
B8: i=4 j=3 s[i]='c' p[j]='c'   | khớp -> i=5, j=4
B9: i=5 j=4 s[i]='d' p[j]='d'   | khớp -> i=6, j=5
Kết quả: True

s = 'acdcb', p = 'a*c?b'
B1: i=0 j=0 s[i]='a' p[j]='a'   | khớp -> i=1, j=1
B2: i=1 j=1 s[i]='c' p[j]='*'   | gặp '*': nhớ star=1, match=1; thử ăn rỗng, j=2
B3: i=1 j=2 s[i]='c' p[j]='c'   | khớp -> i=2, j=3
B4: i=2 j=3 s[i]='d' p[j]='?'   | khớp -> i=3, j=4
B5: i=3 j=4 s[i]='c' p[j]='b'   | kẹt -> lùi: '*' ăn 'c', i=2, j=2
B6: i=2 j=2 s[i]='d' p[j]='c'   | kẹt -> lùi: '*' ăn 'cd', i=3, j=2
B7: i=3 j=2 s[i]='c' p[j]='c'   | khớp -> i

### 8.4. Vì sao chỉ cần nhớ dấu `'*'` **gần nhất**?

Đây là câu hỏi quan trọng nhất của cách này. Khi đã đi qua dấu `'*'` thứ hai, tại sao không bao giờ cần quay lại sửa dấu `'*'` thứ nhất?

**Lập luận "dời sớm hơn vẫn được" (exchange argument):** giả sử mẫu là `X * A * Y`, trong đó `A` là đoạn cố định giữa hai `'*'`.

1. Greedy đặt `A` ở vị trí **sớm nhất** có thể trong `s` (sau phần `X` đã khớp).
2. Giả sử tồn tại một cách khớp đúng nào đó đặt `A` ở vị trí **muộn hơn**.
3. Dời `A` về vị trí sớm nhất mà greedy tìm được: các ký tự bị "bỏ trống" giữa vị trí cũ và vị trí mới chỉ cần cho **dấu `'*'` thứ hai** ăn thêm — mà `'*'` ăn được **mọi thứ**. Phần `Y` phía sau không bị ảnh hưởng.
4. Vậy: **nếu có cách khớp nào đó, thì cũng có cách khớp với `A` ở vị trí sớm nhất.** Cho `'*'` thứ nhất ăn nhiều hơn chỉ đẩy `A` ra muộn hơn — không bao giờ giúp gì thêm.

Nói gọn: một khi đoạn `A` đã được đặt xong, **mọi sự co giãn cần thiết sau đó đều có thể dồn cho `'*'` mới hơn**. Vì thế chỉ cần lưu một cặp `(star, match)` — mỗi khi gặp `'*'` mới, ta ghi đè cặp cũ.

**Vì sao trả `False` đúng?**

- Nhánh 4: chưa từng có `'*'`, nghĩa là phần mẫu đã đi qua là cố định, mỗi ký tự ép đúng 1 ký tự của `s` → không còn lựa chọn nào khác.
- Kết thúc mà `p` còn ký tự khác `'*'`: đoạn `p[star+1:]` (không có `'*'` bên trong cho tới `j`) đã dùng hết phần còn lại của `s`, mà vẫn cần thêm ký tự; cho `'*'` ăn thêm chỉ làm phần còn lại **ngắn hơn** → vô vọng.

### 8.5. Độ phức tạp Cách 5

- **Space:** `O(1)` — chỉ vài biến số nguyên.
- **Time:** thường rất nhanh, nhưng **trường hợp xấu nhất là `O(m · n)`**: mỗi lần quay lui (nhánh 3), `i` lùi về `match` và đoạn sau `'*'` phải so lại từ đầu.

Ví dụ xấu: `s = "aaa...a"` (k chữ `a`), `p = "*" + "a"*(k/2) + "b"`. Mỗi lần thử, đoạn `aaa...a` khớp gần hết rồi mới vấp ở `b`. Cell dưới đếm số vòng lặp `while` so với số ô của bảng DP.

In [19]:
def count_greedy_loops(s: str, p: str) -> int:
    m, n = len(s), len(p)
    i = j = 0
    star, match = -1, 0
    loops = 0
    while i < m:
        loops += 1
        if j < n and (p[j] == '?' or p[j] == s[i]):
            i += 1; j += 1
        elif j < n and p[j] == '*':
            star, match = j, i; j += 1
        elif star != -1:
            j = star + 1; match += 1; i = match
        else:
            break
    return loops

for k in [10, 100, 1000]:
    s = "a" * k
    p = "*" + "a" * (k // 2) + "b"
    print(f"k={k:5}: vòng lặp greedy={count_greedy_loops(s, p):8} | số ô DP={(len(s)+1)*(len(p)+1):8}")

k=   10: vòng lặp greedy=      36 | số ô DP=      88
k=  100: vòng lặp greedy=    2601 | số ô DP=    5353
k= 1000: vòng lặp greedy=  251001 | số ô DP=  503503


Số vòng lặp tăng theo bình phương `k` — cùng bậc với DP. Nhưng với input "bình thường" (ít quay lui), greedy chạy gần tuyến tính và **không tốn bộ nhớ thêm**, nên đây là lời giải rất được ưa chuộng khi phỏng vấn.

## 9. Code nộp LeetCode

### 9.1. Bản chính — Greedy + backtracking (`O(1)` bộ nhớ)

Code nộp bài gọn, không `print`, đúng signature `isMatch(self, s: str, p: str) -> bool`.

In [20]:
class Solution:
    def isMatch(self, s: str, p: str) -> bool:
        m, n = len(s), len(p)
        i = j = 0
        star, match = -1, 0
        while i < m:
            if j < n and (p[j] == '?' or p[j] == s[i]):
                i += 1
                j += 1
            elif j < n and p[j] == '*':
                star, match = j, i
                j += 1
            elif star != -1:
                j = star + 1
                match += 1
                i = match
            else:
                return False
        while j < n and p[j] == '*':
            j += 1
        return j == n

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:32} PASS")

sol = Solution()
tests = [
    ("Example 1", "aa", "a", False),
    ("Example 2", "aa", "*", True),
    ("Example 3", "cb", "?a", False),
    ("cả hai rỗng", "", "", True),
    ("s rỗng, p toàn *", "", "***", True),
    ("s rỗng, p = ?", "", "?", False),
    ("p rỗng, s khác rỗng", "a", "", False),
    ("* ở giữa", "adceb", "*a*b", True),
    ("đuôi không khớp", "acdcb", "a*c?b", False),
    ("* thừa ở cuối", "ab", "ab**", True),
    ("? rồi *", "ab", "?*", True),
    ("quay lui nhiều lần", "abcbcd", "a*bcd", True),
    ("mẫu dài hơn chuỗi", "a", "a?", False),
    ("ví dụ dài", "abefcdgiescdfimde", "ab*cd?i*de", True),
    ("mississippi", "mississippi", "m??*ss*?i*pi", False),
]
for name, s, p, expected in tests:
    check(name, sol.isMatch(s, p), expected)

Example 1                        PASS
Example 2                        PASS
Example 3                        PASS
cả hai rỗng                      PASS
s rỗng, p toàn *                 PASS
s rỗng, p = ?                    PASS
p rỗng, s khác rỗng              PASS
* ở giữa                         PASS
đuôi không khớp                  PASS
* thừa ở cuối                    PASS
? rồi *                          PASS
quay lui nhiều lần               PASS
mẫu dài hơn chuỗi                PASS
ví dụ dài                        PASS
mississippi                      PASS


### 9.2. Bản thay thế — DP 1 chiều

Nếu trong phỏng vấn bạn thấy DP dễ chứng minh hơn, nộp bản này cũng được chấp nhận (`O(m·n)` time, `O(n)` space).

In [22]:
class SolutionDP:
    def isMatch(self, s: str, p: str) -> bool:
        n = len(p)
        prev = [False] * (n + 1)
        prev[0] = True
        for j in range(1, n + 1):
            prev[j] = prev[j - 1] and p[j - 1] == '*'
        for i in range(1, len(s) + 1):
            cur = [False] * (n + 1)
            for j in range(1, n + 1):
                if p[j - 1] == '*':
                    cur[j] = cur[j - 1] or prev[j]
                elif p[j - 1] == '?' or p[j - 1] == s[i - 1]:
                    cur[j] = prev[j - 1]
            prev = cur
        return prev[n]

sol_dp = SolutionDP()
for name, s, p, expected in tests:
    assert sol_dp.isMatch(s, p) == expected, name
print("SolutionDP PASS toàn bộ", len(tests), "test.")

SolutionDP PASS toàn bộ 15 test.


### 9.3. Test đối chiếu chéo trên input ngẫu nhiên

Ta sinh hàng nghìn cặp `(s, p)` nhỏ, chạy **cả 5 cách** và so với một "trọng tài" độc lập: `fnmatch.fnmatchcase` trong thư viện chuẩn (so khớp kiểu tên file của shell, cũng dùng `?` và `*`).

Lưu ý: `fnmatch` còn hiểu `[...]` là tập ký tự, nên ta **chỉ sinh** chữ thường và `?`, `*` — đúng miền input của đề. `random.seed(44)` cố định để mỗi lần chạy ra cùng bộ test.

In [23]:
import random
import fnmatch

random.seed(44)
methods = {
    "recursive": wildcard_recursive,
    "memo": wildcard_memo,
    "dp": wildcard_dp,
    "dp_1d": wildcard_dp_1d,
    "greedy": wildcard_greedy,
    "Solution": Solution().isMatch,
    "SolutionDP": SolutionDP().isMatch,
}
num_true = 0
for _ in range(3000):
    s = "".join(random.choice("ab") for _ in range(random.randint(0, 7)))
    p = "".join(random.choice("ab?*") for _ in range(random.randint(0, 6)))
    expected = fnmatch.fnmatchcase(s, p)
    num_true += expected
    for name, f in methods.items():
        assert f(s, p) == expected, (name, s, p, expected)
print(f"3000 cặp ngẫu nhiên: cả {len(methods)} hàm khớp fnmatch. Số ca True = {num_true}")

3000 cặp ngẫu nhiên: cả 7 hàm khớp fnmatch. Số ca True = 629


## 10. Vì sao đúng? — tổng hợp invariant

- **Cách 1 (đệ quy):** `match(i, j)` trả đúng câu trả lời cho hậu tố `s[i:]`, `p[j:]`. Đúng vì: base case hiển nhiên; ký tự thường/`'?'` chỉ có một cách ăn; `'*'` có đúng 2 khả năng "dừng" hoặc "ăn thêm 1 và ở lại" — phủ hết mọi độ dài. Mỗi lời gọi làm bài nhỏ hơn (`i + j` tăng hoặc `j` tăng), nên luôn dừng.
- **Cách 2 (memo):** cùng các luật; memo chỉ tránh tính lại, không đổi kết quả.
- **Cách 3/4 (DP):** sau khi điền xong hàng `i`, mọi ô `dp[i][0..n]` đúng định nghĩa "tiền tố `s[:i]` khớp tiền tố `p[:j]`", vì mỗi ô chỉ dùng các ô đã đúng (trên, trái, chéo).
- **Cách 5 (greedy):** bất biến — `p[:star]` đã khớp `s[:vị trí '*' bắt đầu ăn]` theo cách đặt sớm nhất; nếu có lời giải, có lời giải mà mọi đoạn trước `'*'` gần nhất được đặt như greedy (lập luận dời sớm ở 8.4). Nên chỉ cần thử các độ dài khác nhau cho `'*'` gần nhất.

## 11. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Đệ quy thuần | thử 2 lựa chọn cho mỗi `'*'` | mũ theo số `'*'` | `O(m + n)` stack | dạy tư duy, quá chậm để nộp |
| 2. Đệ quy + memo | lưu kết quả `(i, j)` | `O(m·n)` | `O(m·n)` | dễ suy ra từ cách 1, có thể `RecursionError` |
| 3. DP bảng | điền `dp[i][j]` theo tiền tố | `O(m·n)` | `O(m·n)` | dễ trace, dễ chứng minh |
| 4. DP 1 chiều | chỉ giữ 2 hàng | `O(m·n)` | `O(n)` | bản DP nên nộp |
| 5. Greedy + backtracking | nhớ `'*'` gần nhất | `O(m·n)` xấu nhất, thường gần `O(m + n)` | `O(1)` | ngắn, nhanh, cần hiểu vì sao đúng |

(`m = len(s)`, `n = len(p)`.)

## 12. Lỗi thường gặp

### Lỗi 1: Quên rằng `'*'` khớp **chuỗi rỗng** khi khởi tạo hàng 0

Sai: chỉ đặt `dp[0][0] = True` rồi bỏ qua vòng `for j` khởi tạo hàng 0. Khi đó `s = ""`, `p = "*"` trả `False` — sai.

Sửa: `dp[0][j] = dp[0][j-1]` nếu `p[j-1] == '*'`.

### Lỗi 2: Greedy quên xử lý `'*'` thừa ở cuối

Sai: sau vòng `while i < m` trả luôn `j == n`. Với `s = "ab"`, `p = "ab*"`: chuỗi hết khi `j = 2`, còn một `'*'` chưa duyệt → trả `False` — sai.

Sửa: thêm `while j < n and p[j] == '*': j += 1` trước khi `return j == n`.

### Lỗi 3: Nhánh 3 của greedy quên tăng `match`

Sai: `j = star + 1; i = match + 1` (không cập nhật `match`). Lần quay lui sau `i` lại về `match + 1` cũ → lặp vô hạn hoặc thử mãi một độ dài. Phải viết `match += 1; i = match`.

### Lỗi 4: Tạo bảng bằng `[[False]*(n+1)]*(m+1)`

Tất cả các hàng dùng chung một list (xem mục 6.2) → kết quả sai một cách khó hiểu.

### Lỗi 5: Lệch chỉ số trong DP

`dp[i][j]` nói về `s[:i]`, `p[:j]` — ký tự cuối là `s[i-1]`, `p[j-1]`. Viết `s[i]`, `p[j]` sẽ so sai ký tự hoặc `IndexError`.

### Lỗi 6: Đệ quy/memo với input dài 2000 → `RecursionError`

Xem mục 5.3. Dùng DP hoặc greedy khi nộp.

Cell dưới chạy thử bản sai của Lỗi 1 và Lỗi 2 để thấy tận mắt.

In [24]:
def wrong_dp_no_row0(s, p):
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True                        # THIẾU: khởi tạo hàng 0 cho '*'
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == '*':
                dp[i][j] = dp[i][j - 1] or dp[i - 1][j]
            elif char_ok(s[i - 1], p[j - 1]):
                dp[i][j] = dp[i - 1][j - 1]
    return dp[m][n]


def wrong_greedy_no_tail(s, p):
    m, n = len(s), len(p)
    i = j = 0
    star, match = -1, 0
    while i < m:
        if j < n and (p[j] == '?' or p[j] == s[i]):
            i += 1; j += 1
        elif j < n and p[j] == '*':
            star, match = j, i; j += 1
        elif star != -1:
            j = star + 1; match += 1; i = match
        else:
            return False
    return j == n                          # THIẾU: bỏ qua các '*' thừa

for s, p in [("", "*"), ("ab", "*ab")]:
    print(f"Lỗi 1  s={s!r:5} p={p!r:6}: sai={wrong_dp_no_row0(s, p)}, đúng={wildcard_dp(s, p)}")
for s, p in [("ab", "ab*"), ("abc", "a*c**")]:
    print(f"Lỗi 2  s={s!r:5} p={p!r:6}: sai={wrong_greedy_no_tail(s, p)}, đúng={wildcard_greedy(s, p)}")

Lỗi 1  s=''    p='*'   : sai=False, đúng=True
Lỗi 1  s='ab'  p='*ab' : sai=False, đúng=True
Lỗi 2  s='ab'  p='ab*' : sai=False, đúng=True
Lỗi 2  s='abc' p='a*c**': sai=False, đúng=True


## 13. Mẹo nhận dạng pattern và bài liên quan

**Dấu hiệu nên nghĩ tới DP 2 chiều `dp[i][j]`:**

- Có **hai chuỗi** cần đối chiếu từng phần (khớp, so sánh, biến đổi).
- Câu hỏi "tiền tố/hậu tố này của A có ... tiền tố/hậu tố kia của B không?" tự nhiên chia nhỏ được.
- Đệ quy thuần có tham số `(i, j)` và bị gọi lặp → thêm memo → chuyển bảng.

**Dấu hiệu có thể dùng greedy như Cách 5:** ký tự đại diện "ăn bao nhiêu cũng được" (như `'*'` ở đây) — khi đó có thể đặt các đoạn cố định sớm nhất và chỉ quay lui tới đại diện gần nhất. Với regex (bài 10), `'*'` gắn với ký tự đứng trước nên **không** dùng được mẹo này.

**Mẹo nhỏ:** gộp nhiều `'*'` liên tiếp thành một (`"a**b"` → `"a*b"`) không đổi nghĩa, giúp đệ quy bớt nhánh.

**Bài liên quan nên làm tiếp:**

| Bài | Liên hệ |
|---|---|
| 10. Regular Expression Matching | cùng khung `dp[i][j]`, nhưng `'*'` lặp ký tự trước nó |
| 72. Edit Distance | `dp[i][j]` trên hai tiền tố, 3 thao tác |
| 1143. Longest Common Subsequence | `dp[i][j]` kinh điển |
| 97. Interleaving String | `dp[i][j]` trên hai chuỗi trộn thành chuỗi thứ ba |
| 115. Distinct Subsequences | DP **đếm số cách** thay vì True/False (giống Bài tập 5 bên dưới) |

## 14. Bài tập

### Bài 1 — Đoán output (không chạy code)

Mỗi dòng trả về `True` hay `False`?

```python
isMatch("", "*****")
isMatch("abc", "?")
isMatch("abc", "*?")
isMatch("ab", "*a")
isMatch("a", "a*?")
```

### Bài 2 — Tính tay bảng DP

Với `s = "abb"`, `p = "a*b"`, hãy tự vẽ bảng `dp` kích thước `4 × 4` (hàng `ε, a, b, b`; cột `ε, a, *, b`) bằng giấy bút, rồi cho biết đáp án `dp[3][3]`. Sau đó so với đáp án cuối notebook.

### Bài 3 — Điền TODO: DP bảng

Hoàn thiện hàm theo đúng 3 luật ở mục 6.1.

In [25]:
def exercise_wildcard_dp(s: str, p: str) -> bool:
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    # TODO 1: dp[0][0] = ?
    # TODO 2: khởi tạo hàng 0 (chuỗi rỗng) cho các '*' ở đầu mẫu
    # TODO 3: hai vòng for i, j; xét p[j-1] == '*' / khớp 1 ký tự / còn lại
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_wildcard_dp("aa", "a") == False
# assert exercise_wildcard_dp("aa", "*") == True
# assert exercise_wildcard_dp("", "**") == True
# assert exercise_wildcard_dp("adceb", "*a*b") == True

### Bài 4 — Tự viết lại Greedy + backtracking

Không nhìn lại mục 8, tự viết hàm dùng `i`, `j`, `star`, `match`. Nhớ 4 nhánh và vòng `while` cuối.

In [26]:
def exercise_wildcard_greedy(s: str, p: str) -> bool:
    # TODO: viết lời giải O(1) bộ nhớ
    pass

# assert exercise_wildcard_greedy("abcbcd", "a*bcd") == True
# assert exercise_wildcard_greedy("acdcb", "a*c?b") == False
# assert exercise_wildcard_greedy("ab", "ab**") == True
# assert exercise_wildcard_greedy("", "?") == False

### Bài 5 — Biến thể: đếm số cách khớp

Một **cách khớp** là một cách gán cho mỗi ký tự của `p` một đoạn con liên tiếp của `s` (chữ/`'?'` → đúng 1 ký tự hợp lệ, `'*'` → đoạn bất kỳ, kể cả rỗng) sao cho nối các đoạn lại theo thứ tự được đúng `s`. Hãy đếm số cách khớp.

Ví dụ: `s = "ab"`, `p = "**"` có **3** cách: (`""`, `"ab"`), (`"a"`, `"b"`), (`"ab"`, `""`). `s = "aa"`, `p = "*a*"` có **2** cách.

In [27]:
def count_matchings(s: str, p: str) -> int:
    # TODO: DP giống mục 6, nhưng ô chứa SỐ NGUYÊN thay vì True/False
    pass

# assert count_matchings("ab", "**") == 3
# assert count_matchings("aa", "*a*") == 2
# assert count_matchings("", "*") == 1
# assert count_matchings("abc", "a?c") == 1
# assert count_matchings("abc", "a?d") == 0

### Bài 6 — Mini challenge: thêm ký tự `'+'`

Mở rộng mẫu với ký tự mới `'+'`: khớp **một hoặc nhiều** ký tự bất kỳ (không được rỗng). Viết `is_match_plus(s, p)` cho mẫu gồm chữ thường, `'?'`, `'*'`, `'+'`.

In [28]:
def is_match_plus(s: str, p: str) -> bool:
    # TODO: có thể tận dụng một hàm đã có không?
    pass

# assert is_match_plus("a", "+") == True
# assert is_match_plus("", "+") == False
# assert is_match_plus("abc", "a+") == True
# assert is_match_plus("a", "a+") == False
# assert is_match_plus("abcd", "+c+") == True

## 15. Gợi ý theo mức

**Bài 3**

- Mức 1: chuỗi rỗng khớp mẫu rỗng; chuỗi rỗng chỉ khớp mẫu toàn `'*'`.
- Mức 2: `'*'` → nhìn ô **trái** (`'*'` rỗng) hoặc ô **trên** (`'*'` ăn thêm `s[i-1]`).
- Mức 3: `if p[j-1] == '*': dp[i][j] = dp[i][j-1] or dp[i-1][j]` / `elif p[j-1] in ('?', s[i-1]): dp[i][j] = dp[i-1][j-1]`.

**Bài 4**

- Mức 1: khớp được thì tiến cả hai; gặp `'*'` thì ghi nhớ; kẹt thì lùi về `'*'` gần nhất.
- Mức 2: ghi nhớ gồm `star = j` và `match = i`; khi lùi: `j = star + 1`, `match += 1`, `i = match`.
- Mức 3: thứ tự nhánh: khớp → `'*'` → lùi → `return False`; sau vòng lặp bỏ qua `'*'` thừa rồi `return j == n`.

**Bài 5**

- Mức 1: lời giải đếm thường giống hệt lời giải True/False, chỉ đổi `or` thành `+`.
- Mức 2: `dp[0][0] = 1`; với `'*'`: `dp[i][j] = dp[i][j-1] + dp[i-1][j]` — tách theo "`'*'` ăn rỗng" và "`'*'` ăn đoạn kết thúc bằng `s[i-1]`".
- Mức 3: hai trường hợp đó không trùng nhau và phủ hết mọi cách, nên cộng lại là đúng.

**Bài 6**

- Mức 1: "một hoặc nhiều ký tự bất kỳ" = "một ký tự bất kỳ" rồi "không hoặc nhiều ký tự bất kỳ".
- Mức 2: `'+'` tương đương `'?*'`.
- Mức 3: `p.replace('+', '?*')` rồi gọi lại lời giải của bài gốc.

## 16. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1:** `True`, `False`, `True`, `False`, `False`.

- `"", "*****"`: mọi `'*'` ăn rỗng.
- `"abc", "?"`: `'?'` chỉ ăn 1 ký tự, còn thừa `"bc"`.
- `"abc", "*?"`: `'*'` ăn `"ab"`, `'?'` ăn `"c"`.
- `"ab", "*a"`: mẫu kết thúc bằng `'a'` nhưng `s` kết thúc bằng `'b'`.
- `"a", "a*?"`: `'?'` cần thêm 1 ký tự sau `'a'` nhưng `s` đã hết.

**Bài 2:** cell dưới in bảng — đáp án `dp[3][3] = True`.

In [29]:
for s, p, expected in [("", "*****", True), ("abc", "?", False), ("abc", "*?", True), ("ab", "*a", False), ("a", "a*?", False)]:
    assert wildcard_greedy(s, p) == expected == wildcard_dp(s, p)
print("Đáp án Bài 1 đúng.")
print()
print_dp_table("abb", "a*b")

Đáp án Bài 1 đúng.

      ε  a  *  b
0 ε   T  .  .  .
1 a   .  T  T  .
2 b   .  .  T  T
3 b   .  .  T  T
Đáp án dp[m][n] = True


In [30]:
def answer_wildcard_dp(s: str, p: str) -> bool:
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for j in range(1, n + 1):
        if p[j - 1] == '*':
            dp[0][j] = dp[0][j - 1]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == '*':
                dp[i][j] = dp[i][j - 1] or dp[i - 1][j]
            elif p[j - 1] == '?' or p[j - 1] == s[i - 1]:
                dp[i][j] = dp[i - 1][j - 1]
    return dp[m][n]


def answer_wildcard_greedy(s: str, p: str) -> bool:
    m, n = len(s), len(p)
    i = j = 0
    star, match = -1, 0
    while i < m:
        if j < n and (p[j] == '?' or p[j] == s[i]):
            i += 1
            j += 1
        elif j < n and p[j] == '*':
            star, match = j, i
            j += 1
        elif star != -1:
            j = star + 1
            match += 1
            i = match
        else:
            return False
    while j < n and p[j] == '*':
        j += 1
    return j == n


def answer_count_matchings(s: str, p: str) -> int:
    m, n = len(s), len(p)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = 1
    for j in range(1, n + 1):
        if p[j - 1] == '*':
            dp[0][j] = dp[0][j - 1]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == '*':
                dp[i][j] = dp[i][j - 1] + dp[i - 1][j]
            elif p[j - 1] == '?' or p[j - 1] == s[i - 1]:
                dp[i][j] = dp[i - 1][j - 1]
    return dp[m][n]


def answer_is_match_plus(s: str, p: str) -> bool:
    return answer_wildcard_greedy(s, p.replace('+', '?*'))


assert answer_wildcard_dp("aa", "a") == False
assert answer_wildcard_dp("aa", "*") == True
assert answer_wildcard_dp("", "**") == True
assert answer_wildcard_dp("adceb", "*a*b") == True

assert answer_wildcard_greedy("abcbcd", "a*bcd") == True
assert answer_wildcard_greedy("acdcb", "a*c?b") == False
assert answer_wildcard_greedy("ab", "ab**") == True
assert answer_wildcard_greedy("", "?") == False

assert answer_count_matchings("ab", "**") == 3
assert answer_count_matchings("aa", "*a*") == 2
assert answer_count_matchings("", "*") == 1
assert answer_count_matchings("abc", "a?c") == 1
assert answer_count_matchings("abc", "a?d") == 0

assert answer_is_match_plus("a", "+") == True
assert answer_is_match_plus("", "+") == False
assert answer_is_match_plus("abc", "a+") == True
assert answer_is_match_plus("a", "a+") == False
assert answer_is_match_plus("abcd", "+c+") == True

# Kiểm chứng thêm: count_matchings > 0 đúng khi và chỉ khi khớp được
random.seed(7)
for _ in range(1000):
    s = "".join(random.choice("ab") for _ in range(random.randint(0, 6)))
    p = "".join(random.choice("ab?*") for _ in range(random.randint(0, 5)))
    assert (answer_count_matchings(s, p) > 0) == fnmatch.fnmatchcase(s, p)

print("Đáp án tham khảo chạy PASS toàn bộ test!")

Đáp án tham khảo chạy PASS toàn bộ test!


## 17. Template nên nhớ

**Greedy + backtracking — nhớ "khớp / gặp sao / lùi về sao / thua", rồi bỏ sao thừa:**

```python
class Solution:
    def isMatch(self, s: str, p: str) -> bool:
        i = j = 0
        star, match = -1, 0
        while i < len(s):
            if j < len(p) and p[j] in ('?', s[i]):   # khớp
                i += 1; j += 1
            elif j < len(p) and p[j] == '*':         # gặp sao: nhớ lại, thử ăn rỗng
                star, match = j, i
                j += 1
            elif star != -1:                         # lùi về sao gần nhất, ăn thêm 1
                j = star + 1
                match += 1
                i = match
            else:                                    # thua
                return False
        while j < len(p) and p[j] == '*':            # sao thừa ăn rỗng
            j += 1
        return j == len(p)
```

**DP — nhớ "sao: trái or trên; ký tự khớp: chéo":**

```text
dp[0][0] = True;  dp[0][j] = dp[0][j-1] nếu p[j-1] == '*'
'*'          : dp[i][j] = dp[i][j-1] or dp[i-1][j]
'?' / khớp   : dp[i][j] = dp[i-1][j-1]
```

## 18. Checklist trước khi submit

- Đã xử lý `s` rỗng và/hoặc `p` rỗng chưa? (`""`/`""` → True, `""`/`"*"` → True, `""`/`"?"` → False, `"a"`/`""` → False)
- DP: đã khởi tạo hàng 0 cho các `'*'` ở đầu mẫu chưa?
- DP: có dùng `s[i-1]`, `p[j-1]` (không phải `s[i]`, `p[j]`) chưa?
- DP: bảng tạo bằng list comprehension, không dùng `[[...]] * m`?
- Greedy: thứ tự 4 nhánh đúng chưa? Nhánh lùi có `match += 1` chưa?
- Greedy: sau vòng lặp có bỏ qua các `'*'` thừa trước khi `return j == n` chưa?
- Không dùng đệ quy sâu với input dài tới 2000 (tránh `RecursionError`).
- Đã chạy lại 3 example của đề và một vài ca `'*'` ở đầu/giữa/cuối chưa?